# C6: rò rỉ văn bản khi chia tập theo trang — huấn luyện lại trên chia cũ, CÓ ĐỐI CHỨNG
Notebook độc lập, không động tới `best_crnn.pth`, `crnn_seed_*.pth` hay `outputs/`. Chỉ ghi vào `checkpoints/ocr_C6/` và `outputs_GOC/extra/`.

**Vì sao không dùng mô hình chốt:** 560/794 dòng của tập kiểm định chia theo trang (bản 12/11–13/11 06:48) nằm trong tập huấn luyện của mô hình chốt, nên đo mô hình chốt trên đó là vô nghĩa.

**Thiết kế (cùng một mô hình, hai tập kiểm định đều chưa từng thấy):**
- `T0` = dòng thuộc *train* ở CẢ HAI bản chia (4.809 dòng) → huấn luyện 5 hạt giống, 25 epoch, lấy **trọng số epoch cuối** (không chọn checkpoint theo tập nào) và lịch học cố định (nhân 0,5 ở epoch 15, 20, 23), để không tập đánh giá nào ảnh hưởng đến huấn luyện.
- `V_rò` = dòng thuộc *val* ở chia theo trang nhưng *train* ở chia theo đoạn (560 dòng): trang khác trang của `T0`, nhưng đoạn văn có thể trùng với `T0`.
- `V_sạch` = *val* của chia theo đoạn (636 dòng): đoạn văn không trùng `T0` theo cách dựng.
- Độ rò rỉ được đo trực tiếp (cửa sổ 5 từ so với văn bản `T0`) trước khi tin so sánh CER.
- Đối chứng nội bộ: trong `V_rò`, so dòng trùng nhiều (từ 50% cửa sổ) với dòng không trùng.
- Tập test 201 dòng **không dùng**.

In [ ]:
from google.colab import drive, userdata, auth
drive.mount("/content/drive"); auth.authenticate_user()
import os, sys, glob, shutil, zipfile, time, json, math, hashlib, subprocess, io, re, random, unicodedata
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

DRIVE_ROOT = "/content/drive/MyDrive/Vietnamese-Handwriting-OCR"
DATA_DIR, CKPT_DIR, OUT_DIR = f"{DRIVE_ROOT}/data", f"{DRIVE_ROOT}/checkpoints", f"{DRIVE_ROOT}/outputs"
GOC = f"{OUT_DIR}_GOC"; EXTRA = f"{GOC}/extra"; CK_C6 = f"{CKPT_DIR}/ocr_C6"
for d in (EXTRA, CK_C6, f"{GOC}/splits_history"): os.makedirs(d, exist_ok=True)

token = userdata.get("GITHUB_TOKEN").strip()
!rm -rf /content/repo
!git clone -q https://{token}@github.com/wickyhien18/Vietnamese-Handwriting-OCR.git /content/repo
!git -C /content/repo checkout -q main
REPO_COMMIT = subprocess.run(["git", "-C", "/content/repo", "log", "-1", "--format=%h %ci"], capture_output=True, text=True).stdout.strip()
sys.path.insert(0, "/content/repo")
!pip -q install editdistance
try:
    from src.common.index_io import read_index
    from src.common.text_utils import normalize_nfc
except Exception as e:
    print("import lỗi, cài requirements:", e)
    !pip -q install -r /content/repo/requirements.txt
    from src.common.index_io import read_index
    from src.common.text_utils import normalize_nfc

LOCAL_ROOT = "/content/uit_hwdb"
if not os.path.isdir(LOCAL_ROOT):
    archive = sorted(glob.glob(f"{DATA_DIR}/uit_hwdb/*.zip"))[0]
    shutil.copy(archive, "/content/uit_hwdb.zip")
    with zipfile.ZipFile("/content/uit_hwdb.zip") as z: z.extractall(LOCAL_ROOT)

svc = build("drive", "v3", cache_discovery=False)
def get_bytes(fid):
    buf = io.BytesIO(); dl = MediaIoBaseDownload(buf, svc.files().get_media(fileId=fid, supportsAllDrives=True)); done = False
    while not done: _, done = dl.next_chunk()
    return buf.getvalue()
sha12 = lambda b: hashlib.sha256(b).hexdigest()[:12]
print("repo", REPO_COMMIT, "| ảnh tại", LOCAL_ROOT)

In [ ]:
# ---------- dựng các tập từ hai bản chia ----------
nfc = lambda s: unicodedata.normalize("NFC", s)
OLD_IDS = {"12/11 14:45": "1oVECtjl39NBkKACY25Oo0A0007Xt7UC4", "13/11 06:48": "194cj8lMANIRpTU31JIGwX6Pd6mjM_wm4"}
old_bytes = {k: get_bytes(v) for k, v in OLD_IDS.items()}
assert len({sha12(b) for b in old_bytes.values()}) == 1, "hai bản chia theo trang phải giống hệt nhau"
old_b = old_bytes["12/11 14:45"]; old_map = json.loads(old_b.decode("utf-8"))
new_b = open(f"{OUT_DIR}/splits.json", "rb").read(); new_map = json.loads(new_b.decode("utf-8"))
print("chia theo trang (cũ):", sha12(old_b), "| chia theo đoạn (hiện tại):", sha12(new_b))
assert sha12(new_b) == "c87df94d8a2b", "splits.json hiện tại không phải bản 13/11 15:06"
for tag, b in (("page_12-11_0648", old_b), ("passage_13-11_1506", new_b)):       # lưu bản sao có tên rõ nghĩa
    open(f"{GOC}/splits_history/splits_{tag}_{sha12(b)}.json", "wb").write(b)

def derive_sets(old, new):
    ids = [i for i in new]                                    # 6.206 dòng trong chỉ mục
    T0     = [i for i in ids if old.get(i) == "train" and new[i] == "train"]
    V_leak = [i for i in ids if old.get(i) == "val"   and new[i] == "train"]
    V_clean= [i for i in ids if new[i] == "val"]
    test   = [i for i in ids if new[i] == "test"]
    return T0, V_leak, V_clean, test
T0_ids, VL_ids, VC_ids, TEST_ids = derive_sets(old_map, new_map)
print("T0", len(T0_ids), "| V_rò", len(VL_ids), "| V_sạch", len(VC_ids), "| test", len(TEST_ids))
assert (len(T0_ids), len(VL_ids), len(VC_ids), len(TEST_ids)) == (4809, 560, 636, 201), "kích thước lệch bảng chéo đã đo"
assert not (set(T0_ids) & (set(VL_ids) | set(VC_ids) | set(TEST_ids))) and not (set(VL_ids) & set(VC_ids))

index = read_index(f"{OUT_DIR}/dataset_index.jsonl"); by_id = {r["id"]: r for r in index}
pick = lambda ids: [by_id[i] for i in ids]
T0, VL, VC = pick(T0_ids), pick(VL_ids), pick(VC_ids)
page_of = lambda i: (i.split("_data_")[0], int(i.rsplit("_", 2)[1]))
assert not ({page_of(i) for i in T0_ids} & ({page_of(i) for i in VL_ids} | {page_of(i) for i in VC_ids})), "T0 và tập kiểm định dùng chung một trang"

# ---------- đo độ rò rỉ văn bản: cửa sổ 5 từ so với văn bản T0 ----------
toks = lambda s: re.sub(r"[^\w\s]", " ", nfc(s).lower()).split()
wins = lambda t, n=5: [tuple(t[i:i + n]) for i in range(len(t) - n + 1)]
T0_windows = set()
for r in T0: T0_windows.update(wins(toks(r["gt_text"])))
def overlap_frac(items):
    out = []
    for r in items:
        w = wins(toks(r["gt_text"])); out.append(None if not w else sum(x in T0_windows for x in w) / len(w))
    return out
OV = {"leak": overlap_frac(VL), "clean": overlap_frac(VC)}
for k, name in (("leak", "V_rò  "), ("clean", "V_sạch")):
    el = [x for x in OV[k] if x is not None]
    print(f"{name}: {len(el)} dòng đủ điều kiện | có ≥1 cửa sổ trùng T0: {100*sum(x > 0 for x in el)/len(el):.1f}% | từ 50% cửa sổ trùng: {100*sum(x >= 0.5 for x in el)/len(el):.1f}%")
print("=> Nếu V_rò không trùng nhiều hơn V_sạch rõ rệt thì thí nghiệm không có ý nghĩa: DỪNG và báo lại.")

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
try:
    import editdistance as _ed
except ImportError:
    _ed = None

vocab = sorted({ch for r in T0 for ch in normalize_nfc(r["gt_text"])})
char2idx = {c: i + 1 for i, c in enumerate(vocab)}; char2idx["<blank>"] = 0
idx2char = {i: c for c, i in char2idx.items()}; num_classes = len(char2idx)
print("num_classes (từ vựng của T0) =", num_classes)

def preprocess_line_image(path, h=32):
    with Image.open(path) as im:
        g = im.convert("L"); w0, h0 = g.size
        nw = max(16, int(round(w0 * (h / float(h0)))))
        a = np.asarray(g.resize((nw, h), Image.Resampling.BILINEAR), dtype=np.float32)
    return (a / 255.0 - 0.5) / 0.5

class HWDBDataset(Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        r = self.items[i]
        a = preprocess_line_image(os.path.join(LOCAL_ROOT, r["image_path"]))
        lab = [char2idx[c] for c in normalize_nfc(r["gt_text"]) if c in char2idx]
        return torch.from_numpy(a).unsqueeze(0), torch.tensor(lab, dtype=torch.long), a.shape[1]

def collate_fn(batch):
    imgs, labs, ws = zip(*batch); mw = max(ws)
    pad = torch.ones((len(imgs), 1, 32, mw), dtype=torch.float32)
    for i, im in enumerate(imgs): pad[i, :, :, :im.shape[2]] = im
    return pad, torch.nn.utils.rnn.pad_sequence(labs, batch_first=True, padding_value=0), \
        torch.tensor([(w // 4) + 1 for w in ws], dtype=torch.long), torch.tensor([len(l) for l in labs], dtype=torch.long)

class CRNN(nn.Module):
    def __init__(self, num_classes, hidden_size=256):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.Conv2d(256, 256, 3, padding=1), nn.ReLU(True), nn.MaxPool2d((2, 2), (2, 1), (0, 1)),
            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(True), nn.MaxPool2d((2, 2), (2, 1), (0, 1)),
            nn.Conv2d(512, 512, 2, 1, 0), nn.BatchNorm2d(512), nn.ReLU(True))
        self.rnn = nn.LSTM(512, hidden_size, num_layers=2, bidirectional=True, batch_first=True, dropout=0.2)
        self.dropout = nn.Dropout(0.2); self.fc = nn.Linear(hidden_size * 2, num_classes)
    def forward(self, x):
        c = self.cnn(x).squeeze(2).permute(0, 2, 1); o, _ = self.rnn(c)
        return F.log_softmax(self.fc(self.dropout(o)), dim=2).permute(1, 0, 2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("device:", device)
assert device.type == "cuda", "Cần GPU: Runtime > Change runtime type > GPU"

def lev(a, b):
    if _ed is not None: return _ed.eval(a, b)
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1): cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]

def greedy_decode(lp, blank=0):
    p = np.argmax(lp, axis=1); out, prev = [], None
    for x in p:
        x = int(x)
        if x != blank and x != prev: out.append(idx2char[x])
        prev = x
    return nfc("".join(out))

def eval_items(model, items):
    loader = DataLoader(HWDBDataset(items), batch_size=32, shuffle=False, collate_fn=collate_fn)
    model.eval(); hyps = []
    with torch.no_grad():
        for imgs, _, il, _ in loader:
            o = model(imgs.to(device)).permute(1, 0, 2).cpu().numpy()
            for b in range(o.shape[0]): hyps.append(greedy_decode(o[b, :min(int(il[b]), o.shape[1])]))
    refs = [nfc(normalize_nfc(r["gt_text"])) for r in items]
    return [lev(a, b) for a, b in zip(refs, hyps)], [len(a) for a in refs]

def set_all_seeds(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
print("Định nghĩa xong.")

In [ ]:
# ---------- huấn luyện 5 hạt giống trên T0 (tự khôi phục nếu mất phiên) ----------
SEEDS = [42, 123, 456, 789, 2026]
res_path = f"{EXTRA}/c6_results.json"
R = json.load(open(res_path, encoding="utf-8")) if os.path.exists(res_path) else {"seeds": {}}
R.update({"design": "T0=train in both splits; V_leak=old val & new train; V_clean=new val; last-epoch weights; MultiStepLR [15,20,23] x0.5; greedy",
          "sizes": {"T0": len(T0), "V_leak": len(VL), "V_clean": len(VC)}, "repo_commit": REPO_COMMIT,
          "old_split_sha": sha12(old_b), "new_split_sha": sha12(new_b), "num_classes": num_classes})
for seed in SEEDS:
    if str(seed) in R["seeds"]:
        print(f"seed {seed}: đã có kết quả, bỏ qua"); continue
    set_all_seeds(seed); g = torch.Generator(); g.manual_seed(seed)
    loader = DataLoader(HWDBDataset(T0), batch_size=32, shuffle=True, collate_fn=collate_fn, generator=g)
    model = CRNN(num_classes).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[15, 20, 23], gamma=0.5)
    crit = nn.CTCLoss(blank=0, zero_infinity=True)
    t0 = time.perf_counter()
    for epoch in range(1, 26):
        model.train(); tot = 0.0
        for imgs, labs, il, tl in loader:
            imgs, labs = imgs.to(device), labs.to(device)
            opt.zero_grad(); out = model(imgs)
            loss = crit(out, labs, torch.clamp(il, max=out.size(0)), tl); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step(); tot += loss.item()
        sched.step()
        if epoch % 5 == 0 or epoch == 25: print(f"  seed {seed} epoch {epoch:2d} | loss {tot/len(loader):.4f}")
    secs = time.perf_counter() - t0
    ck = f"{CK_C6}/c6_T0_seed_{seed}_epoch25.pth"; torch.save(model.state_dict(), ck)
    e_l, n_l = eval_items(model, VL); e_c, n_c = eval_items(model, VC)
    R["seeds"][str(seed)] = {"train_seconds": round(secs, 1), "final_loss": tot / len(loader), "ckpt_sha12": sha12(open(ck, "rb").read()),
                             "leak": {"err": e_l, "len": n_l}, "clean": {"err": e_c, "len": n_c}}
    json.dump(R, open(res_path, "w", encoding="utf-8"))
    print(f"seed {seed}: V_rò CER {100*sum(e_l)/sum(n_l):.2f}% | V_sạch CER {100*sum(e_c)/sum(n_c):.2f}% | {secs/60:.1f} phút")

In [ ]:
# ---------- tổng hợp ----------
import statistics as st
R = json.load(open(f"{EXTRA}/c6_results.json", encoding="utf-8"))
SDS = sorted(R["seeds"], key=int)
cer = lambda e, n: 100 * sum(e) / sum(n)
L = [cer(R["seeds"][s]["leak"]["err"], R["seeds"][s]["leak"]["len"]) for s in SDS]
C = [cer(R["seeds"][s]["clean"]["err"], R["seeds"][s]["clean"]["len"]) for s in SDS]
gap = [a - b for a, b in zip(L, C)]
for s, a, b in zip(SDS, L, C): print(f"seed {s:>4} | V_rò {a:.2f} | V_sạch {b:.2f} | chênh (rò − sạch) {a-b:+.2f}")
ms = lambda v: (st.mean(v), st.stdev(v) if len(v) > 1 else float("nan"))
print(f"\nCER V_rò   {ms(L)[0]:.2f} ± {ms(L)[1]:.2f}\nCER V_sạch {ms(C)[0]:.2f} ± {ms(C)[1]:.2f}\nchênh      {ms(gap)[0]:+.2f} ± {ms(gap)[1]:.2f}  (n={len(SDS)} hạt giống, độ lệch chuẩn mẫu)")

# đối chứng nội bộ trong V_rò: dòng trùng nhiều với T0 và dòng không trùng
major = np.array([(x is not None and x >= 0.5) for x in OV["leak"]]); minor = np.array([(x is not None and x < 0.5) for x in OV["leak"]])
print(f"\nTrong V_rò: {major.sum()} dòng trùng từ 50% cửa sổ, {minor.sum()} dòng không, {len(major)-major.sum()-minor.sum()} dòng quá ngắn")
for name, m in (("trùng nhiều", major), ("không trùng", minor)):
    v = [100 * sum(np.array(R["seeds"][s]["leak"]["err"])[m]) / sum(np.array(R["seeds"][s]["leak"]["len"])[m]) for s in SDS]
    print(f"  {name:12} CER {ms(v)[0]:.2f} ± {ms(v)[1]:.2f}")

# bootstrap theo trang (hai tập lấy mẫu riêng), trung bình lỗi các hạt giống trên từng dòng
rng = np.random.default_rng(0)
def per_page(ids, key):
    e = np.mean([R["seeds"][s][key]["err"] for s in SDS], axis=0); n = np.array(R["seeds"][SDS[0]][key]["len"]); d = {}
    for i, a, b in zip(ids, e, n): d.setdefault(page_of(i), [0.0, 0])[0] += a; d[page_of(i)][1] += b
    return np.array(list(d.values()))
PL, PC = per_page(VL_ids, "leak"), per_page(VC_ids, "clean")
bs = []
for _ in range(5000):
    a = PL[rng.integers(0, len(PL), len(PL))]; b = PC[rng.integers(0, len(PC), len(PC))]
    bs.append(100 * a[:, 0].sum() / a[:, 1].sum() - 100 * b[:, 0].sum() / b[:, 1].sum())
lo, hi = np.percentile(bs, [2.5, 97.5])
print(f"\nchênh CER (rò − sạch), bootstrap theo trang (5000 lần): {np.mean(L)-np.mean(C):+.2f} điểm, KTC 95% [{lo:+.2f}; {hi:+.2f}] | {len(PL)} trang rò, {len(PC)} trang sạch")
print("Dán lại toàn bộ đầu ra của ô này (và ô 2).")